# fisher_exact_p

> 对应代码：`EconometricsCode/code_in_notes/Chap.11/fisher_exact_p.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.11`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.11")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

文件头注释标明本文件用于演示 Fisher 精确检验的 p 值计算。

我们先清空工作环境，并加载 LaLonde 的 NSW 实验数据（美国国家支持工作示范项目），其中 treat 是随机分配的处理组指示变量。

In [ ]:
%%stata
clear all
use ../datasets/Lalonde_nsw/nsw.dta

Fisher 精确检验基于对处理标签的随机重排，为此我们固定随机数种子以保证结果可复现。

In [ ]:
%%stata
set seed 1900

用 su treat 同时得到总样本量 N 和处理组人数 Nt（treat 之和），存入局部宏供后面重排时使用。

In [ ]:
%%stata
// 计算样本量
su treat
local N=r(N)
local Nt=r(sum)

这里通过一个开关宏 y 选择结果变量：除了原始的收入变量 re78 外，还预设了秩（rank）与对数（log）等备选统计量。秩统计量对异常值更稳健，这体现了 Fisher 检验的灵活性——任何关于两组差异的统计量都可以作为检验统计量。

In [ ]:
%%stata
// 统计量的选择
local y "re78"

如果选择 re78rank，我们就调用自编命令 gen_rank 生成收入的秩（并列值取平均秩），再减去秩的平均值 (N+1)/2 使其中心化；中心化只改变检验统计量的位置、不影响其绝对值大小，因此不影响 p 值。

In [ ]:
%%stata
if "`y'"=="re78rank"{
	gen_rank re78, gen(re78rank)
	replace re78rank=re78rank-(`N'+1)/2
}

若选择 log_re78，则对收入加 1 后取对数（加 1 避免零收入无法取对数），用对数化压缩极端值的影响。

In [ ]:
%%stata
else if "`y'"=="log_re78"{
	gen log_re78=log(1+re78)
}

同理为 re75（处理前一年收入）的秩与对数提供备选分支，便于做安慰剂检验或对基期结果做稳健性检查。

In [ ]:
%%stata
else if "`y'"=="re75rank"{
	gen_rank re75, gen(re75rank)
	replace re75rank=re75rank-(`N'+1)/2
}
else if "`y'"=="log_re75"{
	gen log_re78=log(1+re75)
}

接着我们估计真实的检验统计量：用结果变量对 treat 做简单回归，其系数恰等于处理组与控制组的样本均值之差（两组比较时 OLS 系数即差-in-means 估计量）。把该系数存入 test_stat，作为 Fisher 精确检验的观测值。

In [ ]:
%%stata
// 计算两组均值差的绝对值
qui: reg `y' treat
local test_stat=_b[treat]

我们新建一个数据框（frame）reassign_stats，其中只含一个变量 stats，用来存放每次随机重排算出的检验统计量；再设定重排次数 M=10000，并预生成两个辅助变量：reassign_treat 存放重排后的处理标签，reassign_rnum 存放用于随机排序的均匀随机数。

In [ ]:
%%stata
// 使用Frame存储重新分配的检验统计量
frame create reassign_stats stats
local M=10000
gen reassign_treat=.
gen reassign_rnum=.

循环的核心是随机化推断（randomization inference）：每次迭代我们给每个观测生成一个均匀随机数并据此排序，然后令排序后的前 Nt 个观测为重排后的"处理组"——这等价于保持处理组人数不变、对处理标签做一次完全随机重排；再用重排标签回归并把系数（即重排下的均值差）存入 frame。重复 M 次就得到原假设（处理无效果）下检验统计量的精确抽样分布。

In [ ]:
%%stata
forvalues i=1/`M'{
	quietly{
		replace reassign_rnum=runiform()
		sort reassign_rnum
		replace reassign_treat=_n<=`Nt'
		reg `y' reassign_treat
		frame post reassign_stats (_b[reassign_treat])
	}
}

切换到存放重排统计量的 frame，我们计算 Fisher 精确 p 值：定义 d_extreme 指示每次重排的统计量是否比观测值更极端（双侧检验取绝对值比较），对其求均值即得到"随机重排下出现比观测更极端结果"的比例——这正是 Fisher 精确检验的 p 值。最后同时输出观测到的检验统计量与该 p 值。

In [ ]:
%%stata
// 计算p并画图
frame change reassign_stats
gen d_extreme=abs(stats)>abs(`test_stat')
sum d_extreme
di "test stats=" `test_stat' _skip  ", Fisher's exact p=" r(mean)

画出 M 次重排得到的检验统计量直方图，并用竖线标出实际观测到的统计量位置，直观展示观测值在原假设分布中的极端程度。

In [ ]:
%%stata
hist stats, xline(`test_stat')